<a href="https://colab.research.google.com/github/Imran1hp/YOLO-object-detection-model/blob/main/YOLOv8_Object_setection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download the Dataset from google drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path ="/content/drive/MyDrive/Waste_dataset_for_Yolov8"

In [4]:
import os
os.listdir(file_path)

['Waste.yolov8.zip',
 'train',
 'README.roboflow.txt',
 'valid',
 'test',
 'data.yaml']

In [5]:
# import zipfile

# zip_file_path = os.path.join(file_path, 'Waste.yolov8.zip')

# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#     zip_ref.extractall(file_path)
# print(f"Unzipped {zip_file_path} to {file_path}")

In [6]:
os.listdir(file_path)

['Waste.yolov8.zip',
 'train',
 'README.roboflow.txt',
 'valid',
 'test',
 'data.yaml']

In [7]:
data_path =  os.path.join(file_path , 'train')
os.listdir(data_path)

['images', 'labels', 'labels.cache']

In [8]:
data_path

'/content/drive/MyDrive/Waste_dataset_for_Yolov8/train'

# Split and creat the valid and test files

In [9]:
import os
import random
import shutil

random.seed(42)

image_path = data_path +"/images"
label_path = data_path +"/labels"

os.makedirs(file_path + "/valid/images" , exist_ok = True)
os.makedirs(file_path + "/valid/labels" , exist_ok = True)

os.makedirs(file_path + "/test/images" ,exist_ok= True)
os.makedirs(file_path + "/test/labels" , exist_ok = True)

images = os.listdir(image_path)
random.shuffle(images)


train_ratio = 0.7
valid_ratio = 0.2
test_ratio = 0.1
total = len(images)

train_end = int(total * train_ratio)
valid_end = train_end + int(total * valid_ratio)

train_files = images[:train_end]
valid_files = images[train_end:valid_end]
test_files = images[valid_end:]


In [10]:
def move_files(file_list , target_folder):
  for img_file in file_list:

    label_file = os.path.splitext(img_file)[0]+'.txt'

    shutil.move(
        os.path.join(image_path , img_file),
        os.path.join(file_path ,target_folder , 'images',img_file)
    )

    shutil.move(
        os.path.join(label_path , label_file),
        os.path.join(file_path ,  target_folder ,"labels" ,label_file)
    )


#filter the files with proper labels

In [11]:
import os

def filter_files_with_existing_labels(file_list, image_base_path, label_base_path):
    counter_for_skip_files =0;
    filtered_list = []
    for img_file in file_list:
        img_full_path = os.path.join(image_base_path, img_file)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_full_path = os.path.join(label_base_path, label_file)
        if os.path.exists(img_full_path) and os.path.exists(label_full_path):
            filtered_list.append(img_file)
        else:
            counter_for_skip_files = counter_for_skip_files+1
            print(f"Skipping {img_file} as its image or label file is missing.")

    print(f'Total number of images without proper label is {counter_for_skip_files}')
    return filtered_list



filtered_valid_files = filter_files_with_existing_labels(valid_files, image_path, label_path)
filtered_test_files = filter_files_with_existing_labels(test_files, image_path, label_path)


move_files(filtered_valid_files ,"valid" )
move_files(filtered_test_files , "test")

Total number of images without proper label is 0
Total number of images without proper label is 0


In [12]:
data_path

'/content/drive/MyDrive/Waste_dataset_for_Yolov8/train'

In [13]:
!ls /content/drive/MyDrive/Waste_dataset_for_Yolov8/valid/images | wc -l

1276


In [14]:
!ls /content/drive/MyDrive/Waste_dataset_for_Yolov8/valid/labels | wc -l

1276


#Install the YOlO model from Ultralytics

In [15]:
!pip install ultralytics --quiet

In [16]:
from ultralytics import YOLO


In [17]:
model_0 = YOLO('yolov8n.pt')

#Train model_0 with imgsize = 640 , epochs=10

In [18]:
model_0.to('cuda')

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [ ]:
model_0.train(data=os.path.join(file_path, 'data.yaml')
, epochs=10, imgsz=640,
  patience=50)

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Waste_dataset_for_Yolov8/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective=0.0, plots=True, pose=12.0, pretrained=Tr

Now, let's pick a test image and run a prediction using the trained model.

In [ ]:
import os

test_images_path = os.path.join(file_path, 'test', 'images')
test_image_files = os.listdir(test_images_path)

# Choose the first image from the list as an example
if test_image_files:
    test_image_name = test_image_files[0]
    test_image_full_path = os.path.join(test_images_path, test_image_name)
    print(f"Predicting on image: {test_image_full_path}")

    # Run inference on the test image
    results = model_0.predict(source=test_image_full_path, show=True, conf=0.25)
else:
    print("No images found in the test/images directory.")